# NB_R02 · 국면 분류 HMM 학습과 검증 (walk-forward)

**[시장국면분석ML] 2/3 단계.**

- 모델: 가우시안 HMM(대각 공분산, numpy 구현이라 별도 설치 불필요), 상태 3개 → 상승장 / 횡보장 / 폭락장
- 미래 정보 차단: 첫 750거래일로 학습한 뒤, 21거래일마다 그때까지의 자료로만 다시 학습합니다. 각 날짜의 확률은 그날까지의 관측만 쓴 **필터링 확률**입니다. 첫 학습 구간의 라벨은 `initial_in_sample`로 따로 표시합니다.
- 상태 이름: 상태별 (20일 수익률 − 20일 변동성) 점수 순서로 붙입니다. 라벨은 새 국면 확률 0.6 이상이 5일 연속일 때만 바꿉니다.
- 검증: BIC(상태 수), 알려진 하락 구간 포착률, HMM과 GMM의 실루엣 비교, 연간 전환 횟수, 실시간 필터링과 사후 추정의 일치율, KNN 이웃의 국면 일치율
- 출력: Delta `gold_regime_daily`, `Files/regime/runs/<RUN_ID>/model.json · bic.json · validation.json`, (선택) MLflow 실험 `MarketRegime_HMM`

In [ ]:
RUN_ID = ""
LOG_TO_MLFLOW = True

In [ ]:
# 모듈 무결성 확인 후 불러오기 (팀 Gold 노트북과 같은 해시 확인 방식)
import sys, json, hashlib, importlib, time
from pathlib import Path
import numpy as np, pandas as pd
assert Path("/lakehouse/default/Files").is_dir(), "기본 Lakehouse LH_MarketRegime을 연결하세요."
# 팀 원천 Lakehouse(lh_whatif)에는 쓰지 않습니다. 기본 Lakehouse가 lh_whatif이면 여기서 멈춥니다.
try:
    import notebookutils
    _lh = (notebookutils.runtime.context.get("defaultLakehouseName") or "")
except Exception:
    _lh = ""
if _lh.lower() == "lh_whatif" or Path("/lakehouse/default/Files/bronze").exists():
    raise RuntimeError("TEAM_LAKEHOUSE_DETECTED: 기본 Lakehouse를 LH_MarketRegime으로 연결하세요.")
MODULE_DIR = Path("/lakehouse/default/Files/reference/market_regime/v1")
_manifest = json.loads((MODULE_DIR / "manifest.json").read_text(encoding="utf8"))
for _f, _h in _manifest["files"].items():
    if hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest() != _h:
        raise RuntimeError("REGIME_MODULE_HASH_MISMATCH:" + _f)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import hmm_numpy, regime_core as rc, fabric_io as fio
for _m in (hmm_numpy, rc, fio):
    importlib.reload(_m)
try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")   # 팀 Gold 노트북과 같은 기준
except Exception:
    pass
print("market_regime", rc.CORE_VERSION, rc.MODEL_VERSION)

In [ ]:
RUN_ID = fio.resolve_run_id(RUN_ID)
ctx = fio.read_json(fio.run_dir(RUN_ID) / "context.json")
cfg = ctx["config"]
feat = fio.load_table(spark, "gold_regime_features", "trading_date")
num = [c for c in feat.columns if c not in ("date_ok",)]
feat[num] = feat[num].apply(pd.to_numeric, errors="coerce")
print("RUN_ID", RUN_ID, "| features", feat.shape, feat.index.min().date(), "→", feat.index.max().date())

In [ ]:
t0 = time.time()
bic = rc.select_n_states(feat, cfg)
print("BIC (첫 학습 구간):", bic["table"], "| BIC 최적", bic["bic_best"], "| 사용", bic["used"])
daily, model_info = rc.walk_forward_regimes(feat, cfg)
print(f"walk-forward 완료 {time.time()-t0:.1f}s | 재학습 {model_info['n_refits']}회 | 학습 기준 {model_info['trained_through']}")
print(daily["regime_name"].value_counts().to_dict())

In [ ]:
t0 = time.time()
validation = rc.validate(feat, daily, model_info, cfg)
print(f"검증 {time.time()-t0:.1f}s")
print(json.dumps({k: validation[k] for k in ["oos_start", "oos_days", "events", "silhouette", "stability",
                                            "filtered_vs_smoothed_agreement", "knn_regime_agreement"]},
                 ensure_ascii=False, indent=1, default=str))
psum = daily[["p_bull", "p_sideways", "p_bear"]].sum(axis=1)
if not np.allclose(psum, 1.0, atol=1e-6):
    raise RuntimeError("PROBABILITY_SUM_INVALID")
if validation["oos_days"] < 252:
    raise RuntimeError("OUT_OF_SAMPLE_TOO_SHORT")

In [ ]:
# 눈으로 확인: KOSPI와 국면 구간 (폰트 문제를 피하려고 영어 라벨 사용)
import matplotlib.pyplot as plt
col = {"bull": "#2e9e6b", "sideways": "#8f99a6", "bear": "#d64545"}
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(daily.index, daily["kospi_close"], color="#1b2733", lw=1)
code_ = daily["regime_code"].to_numpy(); idx = daily.index
s = 0
for t in range(1, len(code_) + 1):
    if t == len(code_) or code_[t] != code_[s]:
        ax.axvspan(idx[s], idx[min(t, len(code_) - 1)], color=col[code_[s]], alpha=0.18, lw=0); s = t
ax.axvline(pd.Timestamp(model_info["walk_forward_start"]), color="k", ls="--", lw=1)
ax.set_title("KOSPI with regime labels (green=bull, gray=sideways, red=bear; dashed=walk-forward start)")
plt.show()

In [ ]:
fio.save_table(spark, daily.reset_index(), "gold_regime_daily")
rd = fio.run_dir(RUN_ID)
fio.write_json_atomic(rd / "model.json", model_info)
fio.write_json_atomic(rd / "bic.json", bic)
fio.write_json_atomic(rd / "validation.json", rc._clean(validation))
if LOG_TO_MLFLOW:
    try:
        import mlflow
        mlflow.set_experiment("MarketRegime_HMM")
        with mlflow.start_run(run_name=RUN_ID):
            mlflow.log_params({"model_version": model_info["model_version"], "n_states": cfg["n_states"],
                               "features": ",".join(cfg["hmm_features"]), "min_train_days": cfg["min_train_days"],
                               "refit_every": cfg["refit_every"], "label_threshold": cfg["label_threshold"],
                               "label_confirm_days": cfg["label_confirm_days"],
                               "gold_snapshot_version": str(ctx["gold_version"].get("gold_snapshot_version"))})
            m = {"oos_days": validation["oos_days"],
                 "switches_per_year": validation["stability"]["label_switches_per_year"],
                 "filtered_vs_smoothed": validation["filtered_vs_smoothed_agreement"] or 0,
                 "knn_agreement": validation["knn_regime_agreement"]["mean"] or 0,
                 "silhouette_hmm": validation["silhouette"]["hmm"] or 0,
                 "silhouette_gmm": validation["silhouette"]["gmm"] or 0}
            for e in validation["events"]:
                m["event_bear_share_" + e["start"]] = e["bear_share"]
            mlflow.log_metrics(m)
            mlflow.log_dict(rc._clean(validation), "validation.json")
            mlflow.log_dict({k: v for k, v in model_info.items()}, "model.json")
        print("MLflow 기록 완료")
    except Exception as e:
        print("MLflow 기록 생략:", type(e).__name__, str(e)[:200])
print("gold_regime_daily 저장 |", daily.index.max().date(), daily["regime_name"].iloc[-1])